In [1]:
import pandas as pd
import mimetypes

In [2]:
new_ref_date = "20260630"
ref = pd.read_csv(f"../../results/ref/_ref_files_{new_ref_date}.csv.gz")

/tmp/ipykernel_59297/2701872155.py:2: DtypeWarning: Columns (11,12,13,14,16,17,18,19,24,25,26,27,30,31,34) have mixed types. Specify dtype option on import or set low_memory=False.
  ref = pd.read_csv(f"../../results/ref/_ref_files_{new_ref_date}.csv.gz")


In [3]:
#on vérifie que les séparateurs sont bien des slashes dans les chemins
ref['path'] = ref['path'].str.replace("\\", "/")

In [4]:
ref['conservation_statut'].value_counts()

conservation_statut
TRANSFERT_S3_OK                                  2071645
INCONNU                                           135385
CORBEILLE                                          85333
DDE - NE PAS GARDER (DIFFUSION)                    42263
CORBEILLE (DIFFUSION)                              37317
À SUPPRIMER (remplacement par Verif / Tampon)      20276
EN LIGNE - À TRANSFERER ?                          18281
À SUPPRIMER (doublons S3 - az)                      6143
À SUPPRIMER (doublons AZ)                           5454
DDE - NE PAS GARDER (FILE_TYPE)                     2210
DDE - NE PAS GARDER (DOUBLONS AZ)                    866
À SUPPRIMER (RENOMMAGE)                              721
À SUPPRIMER                                          686
À TRANSFERER                                         486
À SUPPRIMER (Tests)                                  412
À SUPPRIMER (FILE_TYPE)                              118
DOUBLON - À VOIR                                      38
Name: count

# Remplissage colonnes

In [5]:
for c in ref.columns:
    l = len(ref[ref[c].isna()])
    if l > 0:
        print(f"Colonne {c} vide : {l}")

Colonne conservation_statut vide : 441
Colonne finding_aid vide : 1820818
Colonne unitid vide : 695428
Colonne osiros_id vide : 717179
Colonne mix_objectIdentifierValue vide : 1187679
Colonne mix_fileSize vide : 1187679
Colonne mix_dateTimeCreated vide : 1187679
Colonne mix_formatName vide : 1187679
Colonne mix_byteOrder vide : 1817198
Colonne mix_compressionScheme vide : 1817203
Colonne mix_imageWidth vide : 1187972
Colonne mix_imageHeight vide : 1187972
Colonne mix_xSamplingFrequency vide : 1264514
Colonne mix_ySamplingFrequency vide : 1264514
Colonne mix_samplingFrequencyUnit vide : 1285202
Colonne mix_colorSpace vide : 1817207
Colonne mix_scanningSoftwareName vide : 1961228
Colonne mix_formatVersion vide : 1875386
Colonne publication_statut vide : 441
Colonne corpus_code vide : 441
Colonne oai_set vide : 717179
Colonne s3_key vide : 351566
Colonne s3_uploaded_date vide : 351566
Colonne s3_bucket vide : 351566


In [23]:
def get_mimetype(filename, fillna="N/A"):
    if filename:
        mimetype, _ = mimetypes.guess_type(filename)
        if fillna:
            if mimetype == None:
                mimetype = fillna
        return mimetype

def get_extension_mimetype(df):
    df['extension'] = df['name'].str.extract(r'.*(\..*)$')
    df['mimetype'] = df['name'].apply(get_mimetype)
    #df['guessed_extension'] = df['mimetype'].apply(get_extension_by_mimetype)

    df['file_type'] = df['mimetype']
    df.loc[df['file_type'] == 'image/jpeg', 'file_type'] = 'jpeg'
    df.loc[df['extension'] == '.jp2', 'file_type'] = 'jpeg2000'
    df.loc[df['file_type'] == 'image/tiff', 'file_type'] = 'tiff'
    df.loc[df['file_type'] == 'text/plain', 'file_type'] = 'ocr txt'
    df.loc[df['file_type'] == 'application/pdf', 'file_type'] = 'pdf'
    df.loc[df['file_type'] == 'application/xml', 'file_type'] = 'ocr xml'
    df.loc[df['extension'] == '.alto', 'file_type'] = 'ocr xml'
    df.loc[df['file_type'].str[:4] == 'vide', 'file_type'] = 'video'
    df.loc[df['extension'] == '.MTS', 'file_type'] = 'video'
    df.loc[df['file_type'].str[:4] == 'audi', 'file_type'] = 'audio'
    df.loc[~df['file_type'].isin(['jpeg', 'jpeg2000', 'tiff', 'pdf', 'ocr txt', 'ocr xml', 'video', 'audio']), 'file_type'] = 'autre'
    return df

ref = get_extension_mimetype(ref)

In [7]:
ref['file_type'].value_counts()

file_type
jpeg        673674
tiff        620516
autre       422049
pdf         338162
ocr txt     334782
ocr xml      33312
audio         7347
video          465
jpeg2000        20
Name: count, dtype: int64

In [6]:
ref.loc[ref['source2s3'].isna(), 'source2s3'] = 'az'
ref['source2s3'].value_counts(dropna=False)

source2s3
az                   2332128
AMR_CADN_378-1000      66984
AMR_CADN_314-1000      19777
UnivLille               6927
AMR_CADN_165-0500       2067
AMR_ARCHIPOP             192
Name: count, dtype: int64

In [7]:
ref.loc[ref['conservation_statut'].isna(), 'conservation_statut'] = 'INCONNU'
ref['conservation_statut'].value_counts(dropna=False)

conservation_statut
TRANSFERT_S3_OK                                  2071645
INCONNU                                           135826
CORBEILLE                                          85333
DDE - NE PAS GARDER (DIFFUSION)                    42263
CORBEILLE (DIFFUSION)                              37317
À SUPPRIMER (remplacement par Verif / Tampon)      20276
EN LIGNE - À TRANSFERER ?                          18281
À SUPPRIMER (doublons S3 - az)                      6143
À SUPPRIMER (doublons AZ)                           5454
DDE - NE PAS GARDER (FILE_TYPE)                     2210
DDE - NE PAS GARDER (DOUBLONS AZ)                    866
À SUPPRIMER (RENOMMAGE)                              721
À SUPPRIMER                                          686
À TRANSFERER                                         486
À SUPPRIMER (Tests)                                  412
À SUPPRIMER (FILE_TYPE)                              118
DOUBLON - À VOIR                                      38
Name: count

In [8]:
ref.loc[ref['publication_statut'].isna(), 'publication_statut'] = 'inconnu'
ref['publication_statut'].value_counts(dropna=False)

publication_statut
oui        992737
jamais     963738
inconnu    471600
Name: count, dtype: int64

In [11]:
ref.loc[ref['corpus_code'].isna(), 'corpus_code'] = '_INCONNU'
ref['corpus_code'].value_counts(dropna=False)

corpus_code
PRA_JRX    595291
PRA_ERT    377078
AMR_EC     358109
PRA_AVE    176340
AMR_DEL    170586
            ...  
MED_NPT         4
AMR_5M          3
AMR_5FI         2
AMR_4M          1
AMR_2F          1
Name: count, Length: 108, dtype: int64

In [12]:
ref.loc[ref['s3_uploaded'].isna(), 's3_uploaded'] = False
ref['s3_uploaded'].value_counts(dropna=False, normalize=True)

s3_uploaded
True     0.851287
False    0.148713
Name: proportion, dtype: float64

In [14]:
#ref = ref.drop(columns=['mimetype'])

# Corpus code

In [12]:
ref.loc[(ref['corpus_code'].isna()) & (ref['path'].str.contains('AMR_3FI')), 'corpus_code'] = 'AMR_3FI'
ref[ref['corpus_code'].isna()]

,name,path,size,last_content_modification_date,last_metadata_modification_date,checksum_md5,uuid,extension,file_type,source2s3,...,mix_colorSpace,mix_scanningSoftwareName,mix_formatVersion,publication_statut,corpus_code,oai_set,s3_key,s3_uploaded,s3_uploaded_date,s3_bucket


# S3

In [13]:
ref.columns

Index(['name', 'path', 'size', 'last_content_modification_date',
       'last_metadata_modification_date', 'checksum_md5', 'uuid', 'extension',
       'file_type', 'source2s3', 'conservation_statut', 'finding_aid',
       'unitid', 'osiros_id', 'mix_objectIdentifierValue', 'mix_fileSize',
       'mix_dateTimeCreated', 'mix_formatName', 'mix_byteOrder',
       'mix_compressionScheme', 'mix_imageWidth', 'mix_imageHeight',
       'mix_xSamplingFrequency', 'mix_ySamplingFrequency',
       'mix_samplingFrequencyUnit', 'mix_colorSpace',
       'mix_scanningSoftwareName', 'mix_formatVersion', 'publication_statut',
       'corpus_code', 'oai_set', 's3_key', 's3_uploaded', 's3_uploaded_date',
       's3_bucket'],
      dtype='object')

In [14]:
for c in ['s3_uploaded_date', 's3_key']:
    c_ = c + "_"
    ref.loc[ref[c].isna(), c_] = False
    ref.loc[~ref[c].isna(), c_] = True

In [17]:
ref.groupby(['source2s3', 'conservation_statut', 's3_key_', 's3_uploaded', 's3_uploaded_date_'], dropna=False)['uuid'].count()

source2s3          conservation_statut                            s3_key_  s3_uploaded  s3_uploaded_date_
AMR_ARCHIPOP       DDE - NE PAS GARDER (FILE_TYPE)                False    False        False                      2
                   TRANSFERT_S3_OK                                True     True         True                     190
AMR_CADN_165-0500  DDE - NE PAS GARDER (DIFFUSION)                False    False        False                    715
                   DDE - NE PAS GARDER (DOUBLONS AZ)              False    False        False                    866
                   À TRANSFERER                                   False    False        False                    486
AMR_CADN_314-1000  DDE - NE PAS GARDER (DIFFUSION)                False    False        False                   8338
                   DDE - NE PAS GARDER (FILE_TYPE)                False    False        False                   2206
                   INCONNU                                        False    

In [16]:
#ref.loc[(ref['s3_key_'] == False) & (ref['s3_uploaded_date_'] == False), 's3_uploaded'] = False
ref.loc[(ref['s3_uploaded'] == True) & (ref['conservation_statut'] != 'TRANSFERT_S3_OK'), 'conservation_statut'] = 'TRANSFERT_S3_OK'

In [21]:
#ref = ref.drop(columns=['s3_key_', 's3_uploaded_date_'])
ref = ref.drop(columns=['Unnamed: 0'])
ref.columns

Index(['name', 'path', 'size', 'last_content_modification_date',
       'last_metadata_modification_date', 'checksum_md5', 'uuid', 'extension',
       'file_type', 'source2s3', 'conservation_statut', 'finding_aid',
       'unitid', 'osiros_id', 'mix_objectIdentifierValue', 'mix_fileSize',
       'mix_dateTimeCreated', 'mix_formatName', 'mix_byteOrder',
       'mix_compressionScheme', 'mix_imageWidth', 'mix_imageHeight',
       'mix_xSamplingFrequency', 'mix_ySamplingFrequency',
       'mix_samplingFrequencyUnit', 'mix_colorSpace',
       'mix_scanningSoftwareName', 'mix_formatVersion', 'publication_statut',
       'corpus_code', 'oai_set', 's3_key', 's3_uploaded', 's3_uploaded_date',
       's3_bucket'],
      dtype='object')

# publication statut 

In [22]:
nc = ['corpus_code', 'oai_set', 'finding_aid', 'unitid', 'osiros_id']
nc_ = [c + "_" for c in nc]
for c in nc:
    c_ = c + "_"
    ref.loc[ref[c].isna(), c_] = False
    ref.loc[~ref[c].isna(), c_] = True    

In [23]:
ref.loc[ ref['publication_statut'].isna(), 'publication_statut'] = 'inconnu'

In [24]:
ref.groupby(['publication_statut', 'finding_aid_', 'unitid_', 'corpus_code_', 'oai_set_', 'osiros_id_'], dropna=False)['uuid'].count()

publication_statut  finding_aid_  unitid_  corpus_code_  oai_set_  osiros_id_
inconnu             False         False    True          False     False         404385
                    True          False    True          False     False          56185
                                  True     True          False     False           9875
                                                         True      True            3652
jamais              False         False    True          False     False         140660
                                  True     True          True      True          642636
                    True          False    True          False     False         109302
                                  True     True          False     False           2983
                                                         True      True           68282
oui                 False         True     True          True      True          643294
                    True          True    

In [25]:
ref.groupby(['publication_statut', 'conservation_statut'], dropna=False)['uuid'].count()

publication_statut  conservation_statut                          
inconnu             INCONNU                                          136332
                    TRANSFERT_S3_OK                                  337279
                    À TRANSFERER                                        486
jamais              CORBEILLE                                         85333
                    CORBEILLE (DIFFUSION)                             37324
                    DDE - NE PAS GARDER (DIFFUSION)                   42263
                    DDE - NE PAS GARDER (DOUBLONS AZ)                   866
                    DDE - NE PAS GARDER (FILE_TYPE)                    2210
                    INCONNU                                            1906
                    TRANSFERT_S3_OK                                  765539
                    À SUPPRIMER                                         686
                    À SUPPRIMER (FILE_TYPE)                             118
                    À 

In [26]:
ref.loc[ ref['conservation_statut'].str.contains('SUPPR'), 'publication_statut'] = 'jamais'
ref.loc[ ref['conservation_statut'].str.contains('CORBEILLE'), 'publication_statut'] = 'jamais'
ref.loc[ ref['conservation_statut'].str.contains('NE PAS GARDER'), 'publication_statut'] = 'jamais'

In [27]:
ref.columns

Index(['name', 'path', 'size', 'last_content_modification_date',
       'last_metadata_modification_date', 'checksum_md5', 'uuid', 'extension',
       'file_type', 'source2s3', 'conservation_statut', 'finding_aid',
       'unitid', 'osiros_id', 'mix_objectIdentifierValue', 'mix_fileSize',
       'mix_dateTimeCreated', 'mix_formatName', 'mix_byteOrder',
       'mix_compressionScheme', 'mix_imageWidth', 'mix_imageHeight',
       'mix_xSamplingFrequency', 'mix_ySamplingFrequency',
       'mix_samplingFrequencyUnit', 'mix_colorSpace',
       'mix_scanningSoftwareName', 'mix_formatVersion', 'publication_statut',
       'corpus_code', 'oai_set', 's3_key', 's3_uploaded', 's3_uploaded_date',
       's3_bucket', 'corpus_code_', 'oai_set_', 'finding_aid_', 'unitid_',
       'osiros_id_'],
      dtype='object')

In [28]:
ref = ref.drop(columns = ['corpus_code_', 'oai_set_', 'finding_aid_', 'unitid_', 'osiros_id_'])
ref.columns

Index(['name', 'path', 'size', 'last_content_modification_date',
       'last_metadata_modification_date', 'checksum_md5', 'uuid', 'extension',
       'file_type', 'source2s3', 'conservation_statut', 'finding_aid',
       'unitid', 'osiros_id', 'mix_objectIdentifierValue', 'mix_fileSize',
       'mix_dateTimeCreated', 'mix_formatName', 'mix_byteOrder',
       'mix_compressionScheme', 'mix_imageWidth', 'mix_imageHeight',
       'mix_xSamplingFrequency', 'mix_ySamplingFrequency',
       'mix_samplingFrequencyUnit', 'mix_colorSpace',
       'mix_scanningSoftwareName', 'mix_formatVersion', 'publication_statut',
       'corpus_code', 'oai_set', 's3_key', 's3_uploaded', 's3_uploaded_date',
       's3_bucket'],
      dtype='object')

In [29]:
#ref[['name', 'path', 'size', 'last_content_modification_date',
       'last_metadata_modification_date', 'checksum_md5', 'uuid', 'extension',
       'file_type', 'source2s3', 'conservation_statut', 'finding_aid',
       'unitid', 'osiros_id', 'mix_objectIdentifierValue', 'mix_fileSize',
       'mix_dateTimeCreated', 'mix_formatName', 'mix_byteOrder',
       'mix_compressionScheme', 'mix_imageWidth', 'mix_imageHeight',
       'mix_xSamplingFrequency', 'mix_ySamplingFrequency',
       'mix_samplingFrequencyUnit', 'mix_colorSpace',
       'mix_scanningSoftwareName', 'mix_formatVersion', 'publication_statut',
       'corpus_code', 'oai_set', 's3_key', 's3_uploaded', 's3_uploaded_date',
       's3_bucket']].to_csv("_ref_files_20260411_new.csv.gz", index=False)

In [27]:
len(ref)

2427378

In [28]:
ref.columns

Index(['name', 'path', 'size', 'last_content_modification_date',
       'last_metadata_modification_date', 'checksum_md5', 'uuid', 'extension',
       'file_type', 'source2s3', 'conservation_statut', 'finding_aid',
       'unitid', 'osiros_id', 'mix_objectIdentifierValue', 'mix_fileSize',
       'mix_dateTimeCreated', 'mix_formatName', 'mix_byteOrder',
       'mix_compressionScheme', 'mix_imageWidth', 'mix_imageHeight',
       'mix_xSamplingFrequency', 'mix_ySamplingFrequency',
       'mix_samplingFrequencyUnit', 'mix_colorSpace',
       'mix_scanningSoftwareName', 'mix_formatVersion', 'publication_statut',
       'corpus_code', 'oai_set', 's3_key', 's3_uploaded', 's3_uploaded_date',
       's3_bucket', 'mimetype'],
      dtype='object')

In [32]:
import hashlib

def calculate_md5(file_path):
    hash_md5 = hashlib.md5()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

# Usage
file_path = "_ref_files_20260411.csv.gz"  # Replace with your file path
md5_checksum = calculate_md5(file_path)
md5_checksum

'fd556fee45aaaec3f4a3c1cb676dce54'

In [31]:
md5_checksum

'904403894bc6d27601afe3e4426d9f19'

# Sauvegarde nouveau fichier ref

In [ ]:
ref.to_csv(f"../../results/ref/_ref_files_{new_ref_date}.csv.gz", index=False)